# ResQ-AI: Full Experiments Notebook

This notebook runs the complete ResQ-AI experiments on Google Colab.

**Requirements:**
- GPU runtime (T4 recommended)
- ~2-4 hours for full experiments

**Change Runtime:**
Runtime → Change runtime type → GPU (T4)

## [NEW] STEP 1a — Mount Google Drive for Checkpoint Backup

Mount your Drive so that checkpoints and result JSON files survive Colab disconnections.
All experiment outputs are saved here after every epoch **and** after every experiment.
Re-running this notebook after a disconnect will skip already-finished experiments automatically.

In [ ]:
# [NEW CELL] Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/resq_ai_checkpoints'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive checkpoint directory: {DRIVE_DIR}')

In [ ]:
# [NEW CELL] Resume detection: check which experiments already completed
# An experiment is considered done if its .done flag exists in DRIVE_DIR.
# Re-run this after any reconnect to restore state.
import os

ALL_EXPERIMENTS = ['e1', 'e2', 'e3', 'e4', 'e5', 'e6', 'e7']

def is_done(exp_id: str) -> bool:
    return os.path.exists(os.path.join(DRIVE_DIR, f'{exp_id}.done'))

def mark_done(exp_id: str):
    open(os.path.join(DRIVE_DIR, f'{exp_id}.done'), 'w').close()
    print(f'  ✓ Marked {exp_id} as complete in Drive.')

def sync_to_drive(local_src: str, remote_dst: str):
    """Copy local_src to remote_dst using shutil (works without gsutil)."""
    import shutil
    os.makedirs(remote_dst, exist_ok=True)
    if os.path.isdir(local_src):
        shutil.copytree(local_src, remote_dst, dirs_exist_ok=True)
    else:
        shutil.copy2(local_src, remote_dst)
    print(f'  Synced {local_src} → {remote_dst}')

print('Resume status:')
for exp in ALL_EXPERIMENTS:
    status = '✓ DONE (will skip)' if is_done(exp) else '○ PENDING'
    print(f'  {exp}: {status}')

# Also restore any previously saved results from Drive back into the working directory
drive_results = os.path.join(DRIVE_DIR, 'results')
if os.path.isdir(drive_results):
    import shutil
    os.makedirs('results', exist_ok=True)
    shutil.copytree(drive_results, 'results', dirs_exist_ok=True)
    print('\nRestored saved results from Drive.')
else:
    print('\nNo previous results found in Drive — starting fresh.')

In [ ]:
# Check GPU
!nvidia-smi

In [ ]:
# Clone repo and install
# ====================================================================
# IMPORTANT: Replace the URL below with your actual GitHub repo URL.
# Example: https://github.com/YOUR_USERNAME/resq-ai.git
# The URL below is a placeholder and MUST be changed before running.
# See PROGRESS.md > NEEDS HUMAN ACTION for details.
# ====================================================================
REPO_URL = 'https://github.com/hkoul2000/resq-ai.git'  # <-- REPLACE WITH YOUR REPO URL
!git clone {REPO_URL} 2>/dev/null || true
%cd resq-ai
!pip install -q -r requirements.txt

In [ ]:
# Run smoke test first to verify setup
!python -m pytest tests/test_smoke.py -v -m smoke --tb=short

## Download Real Sen1Floods11 Dataset

Download the official hand-labeled Sentinel-1/2 chips and splits directly from Google Cloud Storage (~3 GB, 1-2 minutes via Colab internal network).

In [ ]:
# Download splits and hand-labeled imagery
!mkdir -p data/sen1floods11/v1.1/splits/flood_handlabeled
!mkdir -p data/sen1floods11/v1.1/data/flood_events/HandLabeled

!gsutil -m cp -r gs://sen1floods11/v1.1/splits/flood_handlabeled/* data/sen1floods11/v1.1/splits/flood_handlabeled/ 2>/dev/null || true
!gsutil -m cp -r gs://sen1floods11/v1.1/data/flood_events/HandLabeled/* data/sen1floods11/v1.1/data/flood_events/HandLabeled/

# Verify download
!ls -la data/sen1floods11/v1.1/data/flood_events/HandLabeled/

## Run Experiments

In [ ]:
# E1: Main model comparison
if is_done('e1'):
    print('E1 already complete — skipping.')
else:
    print('Running E1: Main Model Comparison ...')
    !python experiments/run_experiments.py --experiment e1 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e1')

In [ ]:
# E2: Ablation studies
if is_done('e2'):
    print('E2 already complete — skipping.')
else:
    print('Running E2: Ablation Studies ...')
    !python experiments/run_experiments.py --experiment e2 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e2')

In [ ]:
# E3: Calibration analysis
if is_done('e3'):
    print('E3 already complete — skipping.')
else:
    print('Running E3: Calibration Analysis ...')
    !python experiments/run_experiments.py --experiment e3 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e3')

In [ ]:
# E4: Conformal prediction
if is_done('e4'):
    print('E4 already complete — skipping.')
else:
    print('Running E4: Split Conformal Prediction ...')
    !python experiments/run_experiments.py --experiment e4 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e4')

In [ ]:
# E5: Robustness tests
if is_done('e5'):
    print('E5 already complete — skipping.')
else:
    print('Running E5: Robustness Tests ...')
    !python experiments/run_experiments.py --experiment e5 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e5')

In [ ]:
# E6: Resource allocation
if is_done('e6'):
    print('E6 already complete — skipping.')
else:
    print('Running E6: Resource Allocation ...')
    !python experiments/run_experiments.py --experiment e6 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e6')

In [ ]:
# E7: Efficiency benchmarks
if is_done('e7'):
    print('E7 already complete — skipping.')
else:
    print('Running E7: Efficiency Benchmarks ...')
    !python experiments/run_experiments.py --experiment e7 --config configs/default.yaml
    sync_to_drive('results', os.path.join(DRIVE_DIR, 'results'))
    mark_done('e7')

## Generate Figures and Tables

In [ ]:
# Automatically generate all tables and figures directly from aggregated empirical results
!python experiments/generate_figures.py
!python experiments/generate_tables.py

# [NEW] Sync final figures and tables to Drive
sync_to_drive('figures', os.path.join(DRIVE_DIR, 'figures'))
sync_to_drive('paper/tables', os.path.join(DRIVE_DIR, 'paper_tables'))
print('All figures and tables synced to Drive.')

In [ ]:
# Display key figures
from IPython.display import Image, display
import glob

for fig in sorted(glob.glob('figures/*.png')):
    print(f'\n--- {fig} ---')
    display(Image(filename=fig, width=600))

## Download Results

In [ ]:
# Zip results for download
!zip -r resq_ai_results.zip results/ figures/ paper/tables/

# [NEW] Also save zip to Drive as final backup
import shutil
shutil.copy2('resq_ai_results.zip', os.path.join(DRIVE_DIR, 'resq_ai_results.zip'))
print(f'Final results zip saved to Drive: {DRIVE_DIR}/resq_ai_results.zip')

from google.colab import files
files.download('resq_ai_results.zip')